# 02 · Data checks

Every data problem in the team plan, measured on the raw files. The numbers here are the ones quoted
in the report's list of problems, so anyone can rerun and confirm them.

Sections follow the four groups, Customer, Product, Promotion and Search.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))   # run from the notebooks folder

import pandas as pd
from weavve import load, paths
from weavve.load import MARKETPLACE, WEBSITE
pd.set_option("display.max_columns", 50)
print("data:", paths.DATA)
print("outputs:", paths.OUT)
import re, json
F = {}
t = load.orders(); s = load.skus(); e = load.discounts(); f = load.loyalty()
t["channel"] = t["Order sales channel"].map({MARKETPLACE: "marketplace", WEBSITE: "website"}).fillna("other")
SKU, QTY, NET = "Product variant SKU at time of sale", "Quantity ordered", "Net sales"

## Customer group, file 3

**Undo rows.** Refunds and cancellations appear as separate undo rows (`Order or sales reversal = reversal`).
Undo rows on cancelled orders go out with the order. The rest are real refunds and get subtracted from their sale.

In [ ]:
F["rows"] = len(t); F["orders"] = t["Order ID"].nunique()
rev = t[t["Order or sales reversal"] == "reversal"]; canc = t["Is canceled order"]
refunds = rev[~rev["Is canceled order"]]
sales = t[(t["Order or sales reversal"] == "order") & ~canc]
F["undo_rows"] = len(rev); F["undo_rows_on_cancelled"] = int(rev["Is canceled order"].sum())
F["refund_rows"] = len(refunds); F["refund_orders"] = refunds["Order ID"].nunique()
F["refund_value"] = round(refunds[NET].sum()); F["refund_value_website"] = round(refunds.loc[refunds.channel == "website", NET].sum())
F["refund_links_to_sale_pct"] = round(refunds["Line item ID"].isin(set(sales["Line item ID"])).mean() * 100, 1)
F["cancelled_rows"] = int(canc.sum()); F["cancelled_orders"] = t.loc[canc, "Order ID"].nunique()
{k: F[k] for k in ["undo_rows", "undo_rows_on_cancelled", "refund_rows", "refund_orders", "refund_value", "refund_value_website", "refund_links_to_sale_pct", "cancelled_rows", "cancelled_orders"]}

**Refund timing.** A refund recorded in a later month than its order would lower the wrong month's sales.

In [ ]:
first_day = t[t["Order or sales reversal"] == "order"].groupby("Order ID")["Day"].min()
lag = refunds.assign(order_day=refunds["Order ID"].map(first_day))
F["refund_later_month_pct"] = round((lag["Day"].dt.to_period("M") != lag["order_day"].dt.to_period("M")).mean() * 100, 1)
F["refund_median_days_later"] = int((lag["Day"] - lag["order_day"]).dt.days.median())
F["refund_later_month_pct"], F["refund_median_days_later"]

**Rows with no product code.** Shipping lines have no code, 0 items and $0. But some rows with no code have items
and money, so they are real sales. Rows with a code, 0 items and money come from edited orders.

In [ ]:
nocode = sales[sales[SKU].isna()]
F["shipping_rows"] = int(((nocode[QTY] == 0) & (nocode[NET] == 0)).sum())
real = nocode[(nocode[QTY] > 0) & (nocode[NET] != 0)]
F["nocode_sale_rows"] = len(real); F["nocode_sale_value"] = round(real[NET].sum())
F["nocode_sale_value_marketplace"] = round(real.loc[real.channel == "marketplace", NET].sum())
z = sales[sales[SKU].notna() & (sales[QTY] == 0) & (sales[NET] != 0)]
F["qty0_rows_with_money"] = len(z); F["qty0_value"] = round(z[NET].sum())
{k: F[k] for k in ["shipping_rows", "nocode_sale_rows", "nocode_sale_value", "nocode_sale_value_marketplace", "qty0_rows_with_money", "qty0_value"]}

**Can customers be followed over time?** For each order, is its Customer ID one seen on an earlier order?
If marketplace IDs almost never repeat, marketplace repeat buying can't be measured.

In [ ]:
oo = sales.groupby("Order ID").agg(cid=("Customer ID", "first"), d=("Day", "min"), ch=("channel", "first")).sort_values("d")
oo["seen_before"] = oo.duplicated("cid")
F["id_seen_before_website_pct"] = round(oo.loc[oo.ch == "website", "seen_before"].mean() * 100, 1)
F["id_seen_before_marketplace_pct"] = round(oo.loc[oo.ch == "marketplace", "seen_before"].mean() * 100, 1)
both = oo.dropna().groupby("cid").ch.agg(lambda v: {"website", "marketplace"} <= set(v))
F["customers_on_both_channels"] = int(both.sum())
{k: F[k] for k in ["id_seen_before_website_pct", "id_seen_before_marketplace_pct", "customers_on_both_channels"]}

**One very large account.** The biggest spender after cleaning, and which channel it buys through.

In [ ]:
clean = t[~canc]                                   # sales plus refunds, cancelled orders removed
per_cust = clean.groupby("Customer ID").agg(orders=("Order ID", "nunique"), spent=(NET, "sum")).sort_values("spent", ascending=False)
top_id = per_cust.index[0]
F["top_account_orders"] = int(per_cust.iloc[0].orders); F["top_account_spent"] = round(per_cust.iloc[0].spent)
F["top_account_channel"] = clean.loc[clean["Customer ID"] == top_id, "Order sales channel"].value_counts().to_dict()
per_cust.head(5)

**Does our order count match Shopify's?** Website customers only. A high match means orders are grouped correctly.

In [ ]:
web = sales[sales.channel == "website"]
ours = web.groupby("Customer ID")["Order ID"].nunique()
shopify = web.groupby("Customer ID")["Customer number of orders"].max()
F["order_count_matches_shopify_pct"] = round((ours == shopify.reindex(ours.index)).mean() * 100, 1)
F["order_count_matches_shopify_pct"]

## Customer group, file 9.1 loyalty

In [ ]:
F["loyalty_rows"] = len(f); F["loyalty_rows_with_order"] = int(f.order_id.notna().sum())
F["loyalty_used_pct"] = round(f.times_used.mean() * 100, 1)
F["loyalty_used_pct_by_source"] = (f.groupby("source").times_used.mean() * 100).round(1).to_dict()
pd.crosstab(f.source, f.coupon_type)

## Product group, file 4

**Missing product codes.** Sold codes that aren't in the product list, and how much can be recovered by reading
the category from the first two parts of the code, like `BR-QT`.

In [ ]:
lines = sales[sales[SKU].notna() & (sales[QTY] > 0)]
missing = lines[~lines[SKU].isin(set(s.Code))].copy()
F["missing_code_line_pct"] = round(len(missing) / len(lines) * 100, 1)
F["missing_code_value"] = round(missing[NET].sum()); F["missing_codes"] = missing[SKU].nunique()
prefix = lambda c: c.str.split("-").str[:2].str.join("-")
missing["pre"] = prefix(missing[SKU]); s["pre"] = prefix(s.Code)
cat_by_pre = s.groupby("pre").Category.agg(lambda v: v.value_counts().index[0])
missing["category_fixed"] = missing.pre.map(cat_by_pre)
F["missing_value_recoverable_pct"] = round(missing.loc[missing.category_fixed.notna(), NET].sum() / missing[NET].sum() * 100, 1)
print({k: F[k] for k in ["missing_code_line_pct", "missing_code_value", "missing_codes", "missing_value_recoverable_pct"]})
(missing.groupby(missing.category_fixed.fillna("(unknown)"))[NET].sum() / missing[NET].sum() * 100).round(1).sort_values(ascending=False)

## Promotion group, file 8

**Discount code families.** Most rows are one-per-person copies of the same campaign.

In [ ]:
name = e.Name.astype(str); fam = pd.Series("other", index=name.index)
for label, pat in [("Klaviyo email code", r"Custom Ecommerce Email"), ("Alia pop-up", r"^\[Alia"), ("Loyalty reward", r"^WHLP"),
                   ("Welcome code", r"(?i)^welcome"), ("Loyalty welcome", r"(?i)loyaltywelcome"), ("Tidio chat", r"(?i)tidio")]:
    fam[name.str.contains(pat) & (fam == "other")] = label
F["discount_rows"] = len(e); F["discount_rows_used"] = int((e["Times Used In Total"] > 0).sum())
F["discount_rows_never_used_pct"] = round((e["Times Used In Total"].fillna(0) == 0).mean() * 100, 1)
fam.value_counts()

**Can order codes be looked up in file 8?** Codes are tidied first (capitals, the email suffix and timestamps removed).

In [ ]:
def tidy(v):
    v = str(v).upper().strip()
    v = re.sub(r"\s*-\s*CUSTOM ECOMMERCE EMAIL$", "", v)
    v = re.sub(r"\s*\(\d{4}-\d{2}-\d{2} \d{2}:\d{2}\)$", "", v)
    return v.strip()
codes = set(e.Name.map(tidy))
used = lines["Discount code"].dropna().astype(str).str.upper().str.strip()
F["order_codes_found_in_file8_pct"] = round(used.isin(codes).mean() * 100, 1)
F["order_codes_found_in_file8_pct"]

## Search group, files 1, 2 and 12

**Search Console.** The same month can appear in two workbooks, and sometimes the two disagree.

In [ ]:
g = load.gsc_site_clicks()
twice = g.groupby("month").filter(lambda d: len(d) > 1)
disagree = twice.groupby("month").filter(lambda d: d.clicks.nunique() > 1)
F["gsc_months"] = g.month.nunique(); F["gsc_months_in_two_files"] = twice.month.nunique(); F["gsc_months_that_disagree"] = disagree.month.nunique()
disagree.sort_values(["month", "workbook"])

**Semrush.** Its Timestamp often falls in a different month from the file name, and Google's AI answer box
shows up from late 2024.

In [ ]:
sem = load.semrush()
F["semrush_timestamp_other_month_pct"] = round((pd.to_datetime(sem.Timestamp).dt.to_period("M") != sem.month.dt.to_period("M")).mean() * 100, 1)
sem["ai"] = sem["SERP Features by Keyword"].fillna("").str.contains("AI overview", case=False)
ai_share = (sem.groupby("month").ai.mean() * 100).round(1)
F["ai_answer_first_month"] = str(ai_share[ai_share > 0].index.min().date())[:7]; F["ai_answer_latest_pct"] = float(ai_share.iloc[-1])
ai_share.tail(12)

**Past Google Ads.** Brand words (containing weavve) inflate the overall click rate, so the non-brand rate is the fair benchmark.

In [ ]:
a = load.ads(); brand = a.kw.str.contains("weavve")
ctr = lambda d: round(d.Clicks.sum() / d["Impr."].sum() * 100, 2)
F["ads_ctr_all"] = ctr(a); F["ads_ctr_brand"] = ctr(a[brand]); F["ads_ctr_nonbrand"] = ctr(a[~brand])
F["ads_cost_per_click"] = round(a.Cost.sum() / a.Clicks.sum(), 2)
F["red_list_words"] = len(load.ads_red_list())
F["negative_words"] = load.ads_negatives()["Negative keyword"].astype(str).str.strip('[]"+').str.lower().nunique()
{k: F[k] for k in ["ads_ctr_all", "ads_ctr_brand", "ads_ctr_nonbrand", "ads_cost_per_click", "red_list_words", "negative_words"]}

## Save every number

In [ ]:
paths.ensure_out()
out = paths.OUT / "data_checks.json"
out.write_text(json.dumps(F, indent=1, default=str))
print("saved", out)
pd.Series({k: v for k, v in F.items() if not isinstance(v, dict)}, name="value").to_frame()